# Structured output: schema + validate + repair

**Session 4 · Track A · local Ollama**

Force JSON, validate with pydantic, and repair on failure.

In [1]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
import json
from pydantic import BaseModel, ValidationError
from utils import ask


In [2]:
class Review(BaseModel):
    label: str
    confidence: float

def extract(text, retries=1):
    prompt = f'Return ONLY JSON {{"label": "positive|negative|neutral", "confidence": 0.0-1.0}} for: "{text}"'
    raw = ask(prompt)
    for _ in range(retries + 1):
        try:
            return Review(**json.loads(raw))
        except (json.JSONDecodeError, ValidationError) as e:
            raw = ask(prompt + f"\n\nYour last output was invalid ({e}). Return valid JSON only.")
    raise ValueError("could not get valid JSON: " + raw[:120])

print(extract("I really love this"))

label='positive' confidence=0.99


### Worked example

A bigger schema (new `aspects` field), a forced failure where the model returns prose, and the repair loop that recovers valid JSON.


In [3]:
# Worked example: extend the schema, force a failure, watch the repair
import json
from pydantic import BaseModel, ValidationError

class Review(BaseModel):
    label: str
    confidence: float
    aspects: list[str]            # NEW field

SCHEMA = '{"label": "positive|negative|neutral", "confidence": 0.0-1.0, "aspects": ["short phrases"]}'

def extract(text, force_prose=False, retries=2):
    ask_for = "Write a paragraph about" if force_prose else f"Return ONLY JSON {SCHEMA} for"
    raw = ask(f'{ask_for}: "{text}"')
    for attempt in range(retries + 1):
        try:
            return Review(**json.loads(raw)), attempt
        except (json.JSONDecodeError, ValidationError) as e:
            raw = ask(f'Return ONLY valid JSON {SCHEMA}. Last output was invalid: {e}\nText: "{text}"')
    raise ValueError("no valid JSON after retries: " + raw[:120])

good, tries = extract("The battery life is amazing but the camera is grainy")
print(good, f"(after {tries} retries)")

forced, tries = extract("I love it", force_prose=True)   # starts as prose -> repair kicks in
print(forced, f"(after {tries} retries)")


label='neutral' confidence=0.75 aspects=['battery life', 'camera'] (after 0 retries)
label='positive' confidence=0.99 aspects=['love it'] (after 1 retries)


## Your turn - vary the example

1. Add another field (e.g. `summary: str`) and update both the schema string and the model.
2. Make it fail a different way (ask for YAML, or a wrong enum) and confirm repair recovers.
3. Log how many retries each input needs. What is your acceptable retry budget?


In [ ]:
# Your variation here - copy the worked example above and change ONE thing, then re-run
